In [ ]:
import numpy as np
from numpy import pi as π
from tqdm.notebook import tqdm, trange
import matplotlib.pyplot as plt
import ufl
import firedrake
from firedrake import (
    Constant, exp, sqrt, inner, grad, div, dx as dζ, ds, dS, jump
)
import irksome
from irksome import Dt

In [ ]:
nz = 16
mesh = firedrake.UnitIntervalMesh(nz)

In [ ]:
family = "DG"  # <- revise later
degree = 0
density_element = firedrake.FiniteElement(family, "interval", degree)
Q = firedrake.FunctionSpace(mesh, density_element)

thickness_element = firedrake.FiniteElement("R", "interval", 0)
R = firedrake.FunctionSpace(mesh, thickness_element)

velocity_element = firedrake.FiniteElement("CG", "interval", degree + 1)
V = firedrake.VectorFunctionSpace(mesh, velocity_element)

See equation 6 in the CFM paper.
I'm using values just for phase 1 of densification to test things.
The coefficient $c_0$ that they use is the accumulation rate in meters of water equivalent, so we need the conversion factor of $\rho_s/\rho_w$ in the rate constant.

In [ ]:
a_s = Constant(0.3)       # m / yr
ρ_s = Constant(350.0)     # kg / m^3
ρ_i = Constant(917.0)     # kg / m^3
ρ_w = Constant(1000.0)    # kg / m^3
T = Constant(243.0)       # °K

def rate_constant(T, ρ_s):
    R = Constant(8.314)   # kJ / (mol °K)
    Q = Constant(10.16)   # kJ / mol
    c_0 = Constant(11.0)  # 1 / m
    return c_0 * exp(-Q / (R * T)) * ρ_s / ρ_w

In [ ]:
def get_test_function(q):
    z, = ufl.algorithms.extract_coefficients(q)
    Z = z.function_space()
    w = firedrake.TestFunction(Z)
    return firedrake.replace(q, {z: w})

In [ ]:
def thickness_equation(**kwargs):
    names = ("thickness", "surface_mass_balance", "basal_mass_balance", "velocity")
    h, a_s, a_b, ω = map(kwargs.get, names)

    g = get_test_function(h)
    mesh = ufl.domain.extract_unique_domain(h)
    ν = firedrake.FacetNormal(mesh)

    F_surf = (-a_s - h * inner(ω, ν)) * g * ds((2,))
    F_bed = (a_b - h * inner(ω, ν)) * g * ds((1,))
    return Dt(h) * g * dζ + F_surf + F_bed

In [ ]:
def velocity_equation(**kwargs):
    names = (
        "velocity",
        "density",
        "thickness",
        "basal_mass_balance",
        "compaction_rate",
    )
    ω, ρ, h, a_b, C = map(kwargs.get, names)
    
    v = get_test_function(ω)

    F_cells = h * (-inner(ω, v.dx(0)) + C(ρ) / ρ * v[0]) * dζ
    F_bed = a_b * v[0] * ds((1,))
    F_surf = h * inner(ω, v) * ds((2,))

    return F_cells + F_bed + F_surf

In [ ]:
def density_equation(**kwargs):
    names = (
        "density",
        "thickness",
        "velocity",
        "surface_density",
        "surface_mass_balance",
    )
    ρ, h, ω, ρ_s, a_s = map(kwargs.get, names)

    ϕ = get_test_function(ρ)
    mesh = ufl.domain.extract_unique_domain(ρ)
    ν = firedrake.FacetNormal(mesh)
    ω_ν = firedrake.max_value(0, inner(ω, ν))
    f = h * ρ * ω_ν

    F_cells = (Dt(h * ρ) * ϕ - inner(h * ρ * ω, grad(ϕ))) * dζ
    F_facets = jump(f) * jump(ϕ) * dS
    F_surf = -ρ_s * a_s * ϕ * ds((2,))
    F_bed = h * ρ * ω_ν * ϕ * ds((1,))

    return F_cells + F_surf + F_bed + F_facets

### Thickness equation

Testing this in isolation for sanity preservation.

In [ ]:
h = firedrake.Function(R)
h.assign(10.0)

t = firedrake.Constant(0.0)
a_0 = Constant(1.0)
δa = Constant(0.5)
a_s = a_0 + δa * firedrake.sin(2 * π * t)
a_b = a_0

fields = {
    "thickness": h,
    "surface_mass_balance": a_s,
    "basal_mass_balance": a_b,
    "velocity": Constant((0.0,)),
}
F = thickness_equation(**fields)

timestep = 1.0 / 12
dt = firedrake.Constant(timestep)

method = irksome.BackwardEuler()
solver = irksome.TimeStepper(F, method, t, dt, h)

final_time = 5.0
num_steps = int(final_time / timestep)
hs = [float(h)]

for step in trange(num_steps):
    solver.advance()
    hs.append(float(h))
    t.assign(t + dt)

In [ ]:
hs = np.array(hs)
fig, ax = plt.subplots()
ax.plot(hs);

### Velocity equation

Testing in isolation for sanity preservation.
Use some assumed firn density and check that the velocity is correct, including the case where there is no conversion of firn to ice at the column base.

In [ ]:
h = Constant(1.0)

a_c = Constant(1.0)
ρ_c = Constant(800.0)
λ = Constant(10.0)
def testing_compaction(ρ):
    return a_c / λ / ρ_c * ρ**2

ζ, = firedrake.SpatialCoordinate(mesh)
ρ = ρ_c * firedrake.exp(-ζ)

ω = firedrake.Function(V)
a_b = Constant(1.0)

fields = {
    "velocity": ω,
    "density": ρ,
    "thickness": h,
    "basal_mass_balance": a_b,
    "compaction_rate": testing_compaction,
}
F = velocity_equation(**fields)

firedrake.solve(F == 0, ω)

In [ ]:
S = firedrake.FunctionSpace(mesh, velocity_element)
ω_0 = firedrake.Function(S).interpolate(ω[0])

fig, ax = plt.subplots()
firedrake.plot(ω_0, axes=ax);

In [ ]:
ω_exact_expr = -a_b / h - a_c / λ * (1 - firedrake.exp(-ζ))
firedrake.norm(ω_0 - ω_exact_expr) / firedrake.norm(ω_0)

### Density equation

Testing in isolation for sanity preservation.
Use a fixed vertical velocity and check that the final profile is correct.

In [ ]:
ρ = firedrake.Function(Q)

ω_0 = Constant(0.1)
δω = Constant(0.05)
ω_expr = -(ω_0 + δω * ζ)
ω = firedrake.as_vector((-(ω_0 + δω * ζ),))

ρ_s = Constant(350.0)
ρ.assign(ρ_s)
a_s = Constant(h * (ω_0 + δω))

fields = {
    "density": ρ,
    "thickness": h,
    "velocity": ω,
    "surface_density": ρ_s,
    "surface_mass_balance": a_s,
}
F = density_equation(**fields)

final_time = 10.0
timestep = 1.0 / 12
t = Constant(0.0)
dt = Constant(timestep)

method = irksome.BackwardEuler()
solver = irksome.TimeStepper(F, method, t, dt, ρ)

In [ ]:
num_steps = int(final_time / timestep)
ρs = [ρ.copy(deepcopy=True)]
for step in trange(num_steps):
    solver.advance()
    ρs.append(ρ.copy(deepcopy=True))

In [ ]:
fig, ax = plt.subplots()
firedrake.plot(ρs[-10], axes=ax);

In [ ]:
ρ_exact = -ρ_s * a_s / (h * ω_expr)
firedrake.norm(ρ - ρ_exact) / firedrake.norm(ρ)

### Coupled thickness, density, and velocity

Let us pray

This is about 30 cm of ice equivalent in accumulation each year.

In [ ]:
a_s = Constant(1.0)
ρ_s = Constant(350.0)

We start converting firn into ice once the density exceeds 800 kg/m${}^3$.
We also need to make a decision about how much mass of firn we can convert into ice per year.
Here I've said that 1 m of ice equivalent can be converted when the firn density is equal to the ice density.

In [ ]:
ρ_c = Constant(800.0)
a_c = Constant(0.5)

def smooth_max(a, b, ϵ):
    return (a + b + firedrake.sqrt((a - b)**2 + ϵ**2)) / 2

def basal_mass_balance(ρ):
    ϵ = Constant(10.0)
    return a_c * smooth_max(0, ρ - ρ_c, ϵ) / (ρ_i - ρ_c)

In [ ]:
h_0 = Constant(1.0)
ω_0 = firedrake.Function(V)

fields = {
    "velocity": ω_0,
    "density": ρ_s,
    "thickness": h_0,
    "basal_mass_balance": Constant(0.0),
    "compaction_rate": lambda ρ: rate_constant(T, ρ_s) * a_s * (ρ_i  - ρ),
}

F = velocity_equation(**fields)
firedrake.solve(F == 0, ω_0)

In [ ]:
ω_ = firedrake.Function(S).interpolate(ω_0[0])
fig, ax = plt.subplots()
firedrake.plot(ω_, axes=ax);

In [ ]:
Z = R * Q * V

z = firedrake.Function(Z)
z.sub(0).assign(h_0)
z.sub(1).assign(ρ_s)
z.sub(2).assign(ω_0)

h, ρ, ω = firedrake.split(z)

In [ ]:
fields = {
    "thickness": h,
    "density": ρ,
    "velocity": ω,
    "surface_density": ρ_s,
    "surface_mass_balance": a_s,
    "basal_mass_balance": basal_mass_balance(ρ),
    "compaction_rate": lambda ρ: rate_constant(T, ρ_s) * a_s * (ρ_i  - ρ),
}

F_h = thickness_equation(**fields)
F_ρ = density_equation(**fields)
F_ω = velocity_equation(**fields)
F = F_h + F_ρ + F_ω

In [ ]:
method = irksome.BackwardEuler()
timestep = 1 / 96
dt = Constant(timestep)
t = Constant(0.0)
params = {
    "solver_parameters": {
        "mat_type": "nest",
        "snes_type": "newtonls",
        "snes_linesearch_type": "bt",
        "snes_linesearch_max_it": 40,
        "ksp_type": "fgmres",
        "pc_type": "fieldsplit",
        "pc_fieldsplit_type": "schur",
        "pc_fieldsplit_schur_fact_type": "full",
        "pc_fieldsplit_0_fields": "1,2",
        "pc_fieldsplit_1_fields": "0",
        "fieldsplit_0": {
            "ksp_type": "gmres",
            "pc_type": "lu",
        },
        "fieldsplit_1": {
            "ksp_type": "gmres",
            "pc_type": "none",
        },
    },
}
solver = irksome.TimeStepper(F, method, t, dt, z, **params)

In [ ]:
final_time = 20.0
num_steps = int(final_time / timestep)
zs = [z.copy(deepcopy=True)]
for step in trange(num_steps):
    solver.advance()
    t.assign(t + dt)
    zs.append(z.copy(deepcopy=True))

In [ ]:
fig, ax = plt.subplots()

ρ = z.subfunctions[1]
firedrake.plot(zs[-1].subfunctions[1], axes=ax);

In [ ]:
hs = np.array([float(z.subfunctions[0]) for z in zs])
fig, ax = plt.subplots()
ax.plot(hs);

In [ ]:
ρs = [z.subfunctions[1] for z in zs]

masses = [h * firedrake.assemble(ρ * dζ) for h, ρ in zip(hs, ρs)]
fig, ax = plt.subplots()
ax.plot(masses);